# Lesson 6: Behavior

**You will learn:** how to describe a behavioral task, the forms an answer can take, and the two ways a model gives one: by replying, or through a readout of its activity.
**Requires:** CPU only, no downloads, under a minute.

A `TaskContext` says what the subject is asked to do. Its `response` says what form the answer takes:

| Response | Answer |
| --- | --- |
| `Choice(labels)` | one label; optionally the implied probability of each label, or a reported confidence |
| `Scale(low, high, step)` | a rating |
| `Text()` | a written answer, kept as written |

Plain viewing is `TaskContext(task_type='passive')`: there is nothing to answer.

In [1]:
import warnings
warnings.filterwarnings('ignore')  # keep the lesson output short

In [2]:
import tempfile, pathlib
import numpy as np
from PIL import Image
from brainscore_core.supported_data_standards.brainio.stimuli import StimulusSet

# Eight small synthetic images: four bright, four dark. They stand in for real stimuli.
folder = pathlib.Path(tempfile.mkdtemp())
rng = np.random.default_rng(0)
rows = []
for i in range(8):
    bright = i % 2 == 0
    low, high = (150, 255) if bright else (0, 100)
    path = folder / f'image_{i}.png'
    Image.fromarray(rng.integers(low, high, (16, 16, 3), dtype=np.uint8)).save(path)
    rows.append({'stimulus_id': f'image_{i}', 'image_path': str(path),
                 'label': 'bright' if bright else 'dark'})

stimuli = StimulusSet(rows)
stimuli.identifier = 'toy-bright-dark'
stimuli.stimulus_paths = dict(zip(stimuli['stimulus_id'], stimuli['image_path']))
stimuli

,stimulus_id,image_path,label
0,image_0,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,bright
1,image_1,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,dark
2,image_2,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,bright
3,image_3,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,dark
4,image_4,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,bright
5,image_5,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,dark
6,image_6,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,bright
7,image_7,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,dark


## A model that replies

`reply(stimulus_row, prompt)` returns the model's raw reply. The task builds the prompt (the instruction plus the response's directive) and parses the reply. The rule below stands in for a model that looks at the image and answers in words.

In [3]:
from brainscore_core import BrainScoreModel, TaskContext
from brainscore_core.responses import Choice, Scale, Text
from brainscore_core.streaming import BufferedSession

def reply(stimulus_row, prompt):
    brightness = np.asarray(Image.open(stimulus_row['image_path'])).mean()
    if 'number' in prompt:
        return str(round(1 + 6 * brightness / 255))
    if 'exactly one of' in prompt:
        return 'bright' if brightness > 127 else 'dark'
    return f'A {"light" if brightness > 127 else "dark"} square of noise.'

judge = BrainScoreModel(identifier='brightness-judge', supported_modalities={'vision'}, reply=reply)
sorted(judge.out_channels)

['behavior',
 'behavior:choice',
 'behavior:confidence',
 'behavior:rating',
 'behavior:text']

In [4]:
def ask(model, task, channel):
    session = BufferedSession(stimulus_set=stimuli, task_context=task,
                              requested_output_channels=[channel])
    model.participate(session)
    model.reset()
    return session.collect(channel)

choice = ask(judge, TaskContext(instruction='Is this image bright or dark?',
                                response=Choice(['bright', 'dark'])), 'behavior:choice')
print(choice['raw_answer'].values, choice.attrs['measure'])

['bright' 'dark' 'bright' 'dark' 'bright' 'dark' 'bright' 'dark'] {'kind': 'reported', 'source': 'reply'}


In [5]:
rating = ask(judge, TaskContext(instruction='How bright is this image?',
                                response=Scale(1, 7, 1)), 'behavior:rating')
print(rating.values)
text = ask(judge, TaskContext(instruction='Describe this image.', response=Text()), 'behavior:text')
print(text.values[:2])

[6. 2. 6. 2. 6. 2. 6. 2.]
['A light square of noise.' 'A dark square of noise.']


A reply that cannot be parsed is kept, marked invalid (`answer_valid`), and credited nothing.

## A model with a readout

A network trained on something else can still take the task: a classifier (or, for ratings, a ridge regression) is fitted on one layer's activity using labeled `fitting_stimuli`. For this lesson the same eight images are used to fit and to test; a real benchmark keeps them separate.

In [6]:
import torch

torch.manual_seed(0)
# A tiny, untrained convolutional network. Its weights are random.
net = torch.nn.Sequential(
    torch.nn.Conv2d(3, 4, kernel_size=3),  # layer '0'
    torch.nn.ReLU(),                       # layer '1'
    torch.nn.AdaptiveAvgPool2d(2),         # layer '2'
    torch.nn.Flatten(),                    # layer '3'
    torch.nn.Linear(16, 8),                # layer '4'
).eval()

def preprocess(paths):
    """Load image files into a (batch, 3, height, width) float array."""
    images = [np.asarray(Image.open(p).convert('RGB'), dtype=np.float32) / 255 for p in paths]
    return np.stack(images).transpose(0, 3, 1, 2)

In [7]:
from brainscore.model_helpers.vision_wrapper import VisionWrapper

cnn = BrainScoreModel(
    identifier='toy-cnn', model=net,
    extractor={'vision': VisionWrapper(identifier='toy-cnn', model=net, preprocessing=preprocess)},
    behavioral_readout_layer='4',
)
task = TaskContext(instruction='Is this image bright or dark?',
                   response=Choice(['bright', 'dark'], probabilities=True),
                   fitting_stimuli=stimuli)
probabilities = ask(cnn, task, 'behavior:probabilities')
print(probabilities.attrs['measure'])
probabilities.values.round(2)

activations:   0%|          | 0/64 [00:00<?, ?it/s]

layer packaging:   0%|          | 0/1 [00:00<?, ?it/s]

activations:   0%|          | 0/64 [00:00<?, ?it/s]

layer packaging:   0%|          | 0/1 [00:00<?, ?it/s]

{'kind': 'implied', 'source': 'readout'}


array([[0.51, 0.49],
       [0.48, 0.52],
       [0.51, 0.49],
       [0.49, 0.51],
       [0.51, 0.49],
       [0.49, 0.51],
       [0.51, 0.49],
       [0.49, 0.51]], dtype=float32)

The same task went to both models. The judge **reported** its answer; the network's answer is **implied** by its activity. `attrs['measure']` says which. Reported confidence and written answers need a model that replies, so a readout-only model cannot take those tasks:

In [8]:
sorted(cnn.out_channels)

['behavior', 'behavior:choice', 'behavior:probabilities', 'behavior:rating']

## Summary

- `TaskContext` says what to do; `response` says what form the answer takes: `Choice`, `Scale`, `Text`, or nothing for passive viewing.
- A model **replies** (reported) or answers through a **readout** or **likelihood** of its internals (implied). Each result says which.
- `prefer_path` picks the path when a model has several; set it whenever your conclusion depends on it.

**Next:** [Lesson 7](07_perturb_a_model.ipynb) changes the model's activity during a run.